In [103]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import (
    HuggingFaceEmbeddings,
    ChatHuggingFace,
    HuggingFaceEndpoint,
)
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [104]:
class CricketState(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int
    sr:float
    bpb:float
    boundary_percent:float
    summary:str
     

In [105]:
def strike_rate(state:CricketState):
   sr= (state['runs']/state['balls'])*100
   state['sr']=sr
   return {'sr':sr}

In [106]:
def ball_boundary(state:CricketState):
        boundary_ball=(state['sixes']+state['fours'])/state['balls']
        state['bpb']=boundary_ball
        return {'bpb':boundary_ball}
        

In [107]:
def percent(state:CricketState):
    boundary_percent=((state['sixes']*6+state['fours']*4)/state['runs'])*100
    state['boundary_percent']=boundary_percent
    return {'boundary_percent':boundary_percent}

In [108]:
def summary(state:CricketState):
        summary=f"""
         Strike Rate-{state['sr']}\n
         Balls per Boundary{state['bpb']}\n
         Boundary Percent{state['boundary_percent']}
        """
        state['summary']=summary
        return state

In [109]:
graph=StateGraph(CricketState)
graph.add_node("strike_rate",strike_rate)
graph.add_node("balls_per_boundry",ball_boundary)
graph.add_node("boundary_percent",percent)
graph.add_node("summary",summary)

graph.add_edge(START,'strike_rate')
graph.add_edge(START,'balls_per_boundry')
graph.add_edge(START,'boundary_percent')

graph.add_edge('strike_rate','summary')
graph.add_edge('balls_per_boundry','summary')
graph.add_edge('boundary_percent','summary')

graph.add_edge('summary',END)
workflow=graph.compile()



In [110]:
initial_state={
    'runs':100,
    'balls':130,
    'sixes':4,
    'fours':8,
    
}
workflow.invoke(initial_state)

{'runs': 100,
 'balls': 130,
 'fours': 8,
 'sixes': 4,
 'sr': 76.92307692307693,
 'bpb': 0.09230769230769231,
 'boundary_percent': 56.00000000000001,
 'summary': '\n         Strike Rate-76.92307692307693\n\n         Balls per Boundary0.09230769230769231\n\n         Boundary Percent56.00000000000001\n        '}